# Fig. 2F, fig. S3G: treatment APA volcano (KO vs control), half-width, lineages stacked

Half-width; excitatory (top) over inhibitory (bottom) so it pairs side-by-side with the
pseudotime volcano (Fig. 2B). x = model-matched adjusted Delta-psi (induced coefficient from W ~ bs(pt,5) + super_batch + induced; computed by `analysis/organoids/perturbation/04_261007-make_adjusted_dpsi.py`), y = -log10(DRIMSeq padj). The adjusted axis shares the LRT covariate adjustment, so direction is consistent with significance.
Doubly-validated reversal genes (CD24/HSP90AA1/TIMM8B) diamond-marked.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment" / "apa-python.yml").is_file())
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()
try:
    from adjustText import adjust_text

    HAS_AT = True
except ImportError:
    HAS_AT = False

PANEL = "fig3a_treatment_volcano"
PANEL_W_PX, PANEL_H_PX = S.HALF_W_PX, S.mm_to_px(
    84
)  # 85 x 84 mm (half-grid, 2 stacked)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)
KB = f"{DATA_ROOT}/3utr/organoids/knoblich/utrcalling"
LINS = [("excitatory", "Excitatory neurons"), ("inhibitory", "Inhibitory neurons")]
PADJ_THR = 0.05
DOUBLY = ["CD24", "HSP90AA1", "TIMM8B"]

In [ ]:
treat = {
    lin: pd.read_csv(
        f"{KB}/dirmult_treatment_results/{lin}/main_only/drimseq_gene_{lin}_treatment_main.csv"
    )
    for lin, _ in LINS
}
# x-axis = model-matched adjusted Delta-psi: the 'induced' coefficient of
#   W_unit ~ bs(pt, df=5) + super_batch + induced   (WLS, weights = n_cells),
# i.e. the treatment effect on 3'UTR length holding pseudotime and super_batch fixed -
# the same covariate adjustment as the DRIMSeq LRT that supplies the p-values.
# The raw `delta_psi_treatment` is an unadjusted marginal difference,
# confounded by the induced<->super_batch association (super_batch 137 is 100% induced)
# and by the induced-organoid shift toward progenitor cell types. See
# analysis/organoids/perturbation/04_261007-make_adjusted_dpsi.py.
for lin, _ in LINS:
    t = treat[lin]
    adj = pd.read_csv(
        f"{KB}/dirmult_treatment_results/{lin}/main_only/delta_psi_treatment_adjusted_{lin}.csv"
    )
    t = t.merge(
        adj[["gene_id", "delta_psi_treatment_adjusted"]], on="gene_id", how="left"
    )
    t["delta_psi_treatment_raw"] = t["delta_psi_treatment"]
    # use adjusted where available, fall back to raw for the rare gene lacking a WLS fit
    t["delta_psi_treatment"] = t["delta_psi_treatment_adjusted"].fillna(
        t["delta_psi_treatment_raw"]
    )
    t["nl"] = -np.log10(t["adj_pvalue"].clip(lower=1e-300))
    t["sig"] = np.where(
        (t["adj_pvalue"] < PADJ_THR) & (t["delta_psi_treatment"] < 0),
        "short",
        np.where(
            (t["adj_pvalue"] < PADJ_THR) & (t["delta_psi_treatment"] > 0), "long", "NS"
        ),
    )
    treat[lin] = t
    ns = int((t.sig == "short").sum())
    nl = int((t.sig == "long").sum())
    print(
        f"{lin}: sig={ns + nl}  short={ns} long={nl} ({100*ns/max(ns+nl,1):.0f}% short)"
    )

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=FIGSIZE)
for ax, (lin, linlabel) in zip(axes, LINS):
    t = treat[lin]
    for cls, col, z, s in [
        ("NS", S.COL_NS, 1, 3),
        ("short", S.COL_SHORT, 2, 9),
        ("long", S.COL_LONG, 2, 9),
    ]:
        m = (t["sig"] == cls).values
        ax.scatter(
            t.loc[m, "delta_psi_treatment"],
            t.loc[m, "nl"],
            c=col,
            s=s,
            alpha=0.3 if cls == "NS" else 0.8,
            edgecolors="none",
            rasterized=True,
            zorder=z,
        )
    ax.axhline(-np.log10(PADJ_THR), color="gray", ls="--", lw=0.5, zorder=0)
    ax.axvline(0, color="gray", ls=":", lw=0.4, zorder=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.20)
    dv = t[t["gene_id"].isin(DOUBLY)]
    ax.scatter(
        dv["delta_psi_treatment"],
        dv["nl"],
        s=20,
        marker="D",
        facecolors="none",
        edgecolors="black",
        linewidths=0.6,
        zorder=6,
    )
    lab_genes = set(DOUBLY) | set(
        t[t["sig"] != "NS"].nsmallest(3, "adj_pvalue")["gene_id"]
    )
    texts = []
    for _, r in t[t["gene_id"].isin(lab_genes)].iterrows():
        bold = r["gene_id"] in DOUBLY
        texts.append(
            ax.text(
                r["delta_psi_treatment"],
                r["nl"],
                r["gene_id"],
                fontsize=S.FONT_ANNOT - 1,
                fontstyle="italic",
                fontweight="bold" if bold else "normal",
                color="black" if bold else "#4d4d4d",
                zorder=10,
            )
        )
    if HAS_AT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
    ns = int((t["sig"] == "short").sum())
    nl_ = int((t["sig"] == "long").sum())
    ax.text(
        0.5,
        0.96,
        f"{ns + nl_} sig  ({ns}/{nl_})",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 0.5,
        va="top",
        ha="center",
        fontweight="bold",
    )
    ax.set_title(
        linlabel,
        fontsize=S.FONT_TICK,
        color=S.paper_palette_lineage[lin],
        fontweight="bold",
        pad=1,
    )
    ax.set_ylabel(r"$-\log_{10}p_{\mathrm{adj}}$", fontsize=S.FONT_TICK)
    ax.tick_params(labelsize=S.FONT_TICK - 1)
axes[1].set_xlabel(
    r"adjusted $\Delta\psi$ (induced $-$ uninduced)", fontsize=S.FONT_TICK
)
leg = [
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.COL_SHORT,
        markersize=3.5,
        label="shortening",
    ),
    Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=S.COL_LONG,
        markersize=3.5,
        label="lengthening",
    ),
    Line2D(
        [0],
        [0],
        marker="D",
        color="w",
        markerfacecolor="none",
        markeredgecolor="black",
        markersize=4.5,
        label="doubly-valid.",
    ),
]
axes[0].legend(
    handles=leg,
    fontsize=S.FONT_LEGEND - 1,
    frameon=False,
    loc="lower center",
    bbox_to_anchor=(0.5, 1.10),
    ncol=3,
    handletextpad=0.15,
    columnspacing=0.5,
)
fig.subplots_adjust(left=0.15, right=0.97, top=0.88, bottom=0.10, hspace=0.42)
S.save_and_show(fig, PANEL, figure="Fig3")

### Variants: lineage-split treatment volcano (half-width; quarter & eighth height)

Single-lineage versions of the treatment volcano (excitatory and inhibitory separately),
at quarter- and eighth-height, to pair with the single-lineage development panels. Saved as
`fig3a_treatment_volcano__{exc,inh}_{quarter,eighth}`.


In [ ]:
# --- Lineage-split treatment volcano variants (half-width; quarter & eighth height) ---
# Single-lineage (EXC / INH) versions of the treatment volcano. Reuses `treat`, DOUBLY,
# PADJ_THR, LINS, HAS_AT / adjust_text, Line2D.
def _volcano3a(lin, figsize, save_name, *, compact):
    t = treat[lin]
    linlabel = dict(LINS)[lin]
    fig, ax = plt.subplots(figsize=figsize)
    for cls, col, z, s in [
        ("NS", S.COL_NS, 1, 3),
        ("short", S.COL_SHORT, 2, 9),
        ("long", S.COL_LONG, 2, 9),
    ]:
        m = (t["sig"] == cls).values
        ax.scatter(
            t.loc[m, "delta_psi_treatment"],
            t.loc[m, "nl"],
            c=col,
            s=s,
            alpha=0.3 if cls == "NS" else 0.8,
            edgecolors="none",
            rasterized=True,
            zorder=z,
        )
    ax.axhline(-np.log10(PADJ_THR), color="gray", ls="--", lw=0.5, zorder=0)
    ax.axvline(0, color="gray", ls=":", lw=0.4, zorder=0)
    ax.set_ylim(top=ax.get_ylim()[1] * 1.20)
    dv = t[t["gene_id"].isin(DOUBLY)]
    ax.scatter(
        dv["delta_psi_treatment"],
        dv["nl"],
        s=20,
        marker="D",
        facecolors="none",
        edgecolors="black",
        linewidths=0.6,
        zorder=6,
    )
    nlab = 2 if compact else 3
    lab = set(DOUBLY) | set(
        t[t["sig"] != "NS"].nsmallest(nlab, "adj_pvalue")["gene_id"]
    )
    texts = []
    for _, r in t[t["gene_id"].isin(lab)].iterrows():
        b = r["gene_id"] in DOUBLY
        texts.append(
            ax.text(
                r["delta_psi_treatment"],
                r["nl"],
                r["gene_id"],
                fontsize=S.FONT_ANNOT - 1,
                fontstyle="italic",
                fontweight="bold" if b else "normal",
                color="black" if b else "#4d4d4d",
                zorder=10,
            )
        )
    if HAS_AT and texts:
        adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle="-", color="gray", lw=0.3))
    ns = int((t["sig"] == "short").sum())
    nl_ = int((t["sig"] == "long").sum())
    ax.text(
        0.5,
        0.965,
        f"{ns + nl_} sig  ({ns}/{nl_})",
        transform=ax.transAxes,
        fontsize=S.FONT_ANNOT - 0.5,
        va="top",
        ha="center",
        fontweight="bold",
    )
    ax.set_ylabel(r"$-\log_{10}p_{\mathrm{adj}}$", fontsize=S.FONT_TICK)
    ax.set_xlabel(
        r"adjusted $\Delta\psi$ (induced $-$ uninduced)", fontsize=S.FONT_TICK
    )
    ax.tick_params(labelsize=S.FONT_TICK - 1)
    leg = [
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_SHORT,
            markersize=3.5,
            label="shortening",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            color="w",
            markerfacecolor=S.COL_LONG,
            markersize=3.5,
            label="lengthening",
        ),
        Line2D(
            [0],
            [0],
            marker="D",
            color="w",
            markerfacecolor="none",
            markeredgecolor="black",
            markersize=4.5,
            label="doubly-valid.",
        ),
    ]
    if compact:
        ax.legend(
            handles=leg,
            fontsize=S.FONT_LEGEND - 1.5,
            frameon=False,
            loc="upper left",
            bbox_to_anchor=(0.0, 0.99),
            handletextpad=0.1,
            labelspacing=0.15,
            borderpad=0.1,
        )
        fig.subplots_adjust(left=0.14, right=0.97, top=0.965, bottom=0.30)
    else:
        ax.legend(
            handles=leg,
            fontsize=S.FONT_LEGEND - 1,
            frameon=False,
            loc="lower center",
            bbox_to_anchor=(0.5, 1.01),
            ncol=3,
            handletextpad=0.15,
            columnspacing=0.5,
        )
        ax.set_title(
            linlabel,
            fontsize=S.FONT_TICK,
            color=S.paper_palette_lineage[lin],
            fontweight="bold",
            pad=14,
        )
        fig.subplots_adjust(left=0.15, right=0.97, top=0.80, bottom=0.155)
    S.save_and_show(fig, save_name, figure="Fig3")


for _lin in ("excitatory", "inhibitory"):
    _volcano3a(
        _lin,
        (S.HALF_W_IN, S.QUARTER_H_IN),
        f"fig3a_treatment_volcano__{_lin[:3]}_quarter",
        compact=False,
    )
    _volcano3a(
        _lin,
        (S.HALF_W_IN, S.EIGHTH_H_IN),
        f"fig3a_treatment_volcano__{_lin[:3]}_eighth",
        compact=True,
    )